# 🧠 A.R.C. — Adaptive Reasoning Chain v4
### AIMO Progress Prize 3 · Diversified Rollouts, Falsification, and Final Judging

Built on **GPT-OSS-120B**

This notebook uses a multi-stage inference system instead of plain majority voting.

1. It runs diversified rollouts with three reasoning styles: `proof`, `compute`, and `hybrid`.
2. It groups candidate answers by agreement across these different styles.
3. It sends the top candidates through a strict verifier that tries to reject each answer first.
4. If verification is still ambiguous, it uses a final low-temperature judge over the best traces.

The goal is to reward answers that are not only popular, but also survive independent falsification attempts.

In [1]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

Found existing installation: keras 3.10.0
Uninstalling keras-3.10.0:
  Successfully uninstalled keras-3.10.0
Found existing installation: matplotlib 3.10.0
Uninstalling matplotlib-3.10.0:
  Successfully uninstalled matplotlib-3.10.0
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.simplefilter('ignore')

In [3]:
import os
import sys
import subprocess

In [4]:
def set_env(wheels_dir):
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', 
        '--no-index', '--find-links', wheels_dir, 
        'unsloth', 'trl', 'vllm', 'openai_harmony'
    ], check=True)

In [5]:
set_env(wheels_dir='/kaggle/input/datasets/chandan989/arc-aimo-utils/wheels')

Looking in links: /kaggle/input/datasets/chandan989/arc-aimo-utils/wheels
Processing /kaggle/input/datasets/chandan989/arc-aimo-utils/wheels/unsloth-2025.12.9-py3-none-any.whl
Processing /kaggle/input/datasets/chandan989/arc-aimo-utils/wheels/trl-0.24.0-py3-none-any.whl
Processing /kaggle/input/datasets/chandan989/arc-aimo-utils/wheels/vllm-0.11.2-cp38-abi3-manylinux1_x86_64.whl
Processing /kaggle/input/datasets/chandan989/arc-aimo-utils/wheels/openai_harmony-0.0.8-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/input/datasets/chandan989/arc-aimo-utils/wheels/unsloth_zoo-2025.12.7-py3-none-any.whl (from unsloth)
Processing /kaggle/input/datasets/chandan989/arc-aimo-utils/wheels/tyro-1.0.3-py3-none-any.whl (from unsloth)
Processing /kaggle/input/datasets/chandan989/arc-aimo-utils/wheels/xformers-0.0.33.post1-cp39-abi3-manylinux_2_28_x86_64.whl (from unsloth)
Processing /kaggle/input/datasets/chandan989/arc-aimo-utils/wheels/bitsandbytes-0.49.0-py3-none-manyli

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyldavis 3.4.1 requires scikit-learn>=1.0.0, which is not installed.
tpot 1.1.0 requires matplotlib>=3.6.2, which is not installed.
tpot 1.1.0 requires scikit-learn>=1.6, which is not installed.
ydata-profiling 4.18.1 requires matplotlib<=3.10,>=3.5, which is not installed.
librosa 0.11.0 requires scikit-learn>=1.1.0, which is not installed.
cuml-cu12 26.2.0 requires scikit-learn>=1.5, which is not installed.
shap 0.50.0 requires scikit-learn, which is not installed.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
bigframes 2.35.0 requires matplotlib>=3.7.1, which is not installed.
giddy 2.3.8 requires matplotlib, which is not installed.
sentence-transformers 5.2.3 requires scikit-learn, which is not installed.
pynndescent 0.6.0 requires scikit-learn>=0.18, which is 

In [6]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/input/datasets/chandan989/arc-aimo-utils/tiktoken_encodings'

In [7]:
import gc
import re
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [8]:
class ARCConfig:
    
    proof_prompt = (
        'You are a world-class IMO medalist. Prefer elegant structural reasoning, invariants, '
        'extremal arguments, and exact derivations. Use Python only to validate a derived claim. '
        'Final answer must be an integer from 0 to 99999 in \\boxed{}.'
    )
    
    compute_prompt = (
        'You are a world-class olympiad problem solver. Use Python early whenever brute force on '
        'small cases, modular search, symbolic algebra, or pattern testing can eliminate bad ideas. '
        'Final answer must be an integer from 0 to 99999 in \\boxed{}.'
    )
    
    hybrid_prompt = (
        'You are a world-class IMO competitor. Mix proof ideas with short computational checks. '
        'If one line of attack stalls, restart from a different viewpoint instead of forcing it. '
        'Final answer must be an integer from 0 to 99999 in \\boxed{}.'
    )
    
    verifier_prompt = (
        'You are a strict mathematical verifier. Your job is to test one candidate answer, trying '
        'to falsify it before you support it. Use Python aggressively for counterexamples, exact '
        'checks, and exhaustive search on reduced cases. End with exactly one line: '
        'VERDICT: SUPPORT, VERDICT: REJECT, or VERDICT: UNSURE. If and only if you support the '
        'candidate, also include the supported value in \\boxed{}.'
    )
    
    tool_prompt = (
        'Use Python for exact algebra, modular arithmetic, exhaustive checks on small cases, '
        'symbolic manipulation, and sanity checks. State what you are testing, keep code concise, '
        'and always print the decisive result.'
    )
    
    preference_prompt = (
        'You have access to math, numpy, and sympy. Prefer sympy for exact algebra and number '
        'theory. When a conjecture can be stress-tested on small instances, do that early. '
        'Before boxing an answer, try to rule out at least one plausible wrong alternative.'
    )
    
    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1'
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17400
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 6
    sandbox_timeout = 3

    stream_interval = 200
    context_tokens = 65536
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 256
    early_stop = 4
    attempts = 8
    workers = 16
    turns = 128
    seed = 42

    finalists = 3
    enable_verifier = True
    enable_judge = True
    verifier_passes = 2
    verifier_temperature = 0.2
    verifier_turns = 64
    judge_temperature = 0.1
    judge_tokens = 512
    trace_excerpt_chars = 2400
    min_attempts_before_stop = 5
    selection_time_cap = 90
    selection_time_floor = 45
    selection_time_fraction = 0.12

    gpu_memory_utilization = 0.96
    temperature = 1.0
    min_p = 0.02

In [9]:
set_seed(ARCConfig.seed)

In [10]:
class ARCTemplate:

    def __init__(self):

        pass

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:

        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:

        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)

        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, user_message]

In [11]:
class ARCSandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):
        
        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):

        self.close()

In [12]:
class ARCTool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        
        self._owns_session = sandbox is None
        
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = ARCSandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:

        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction, 
            tools=[]
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:

        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')

        if channel:
            message = message.with_channel(channel)

        return message

    def process_sync_plus(self, message: Message) -> list[Message]:

        self._ensure_session()
        raw_script = message.content[0].text
        final_script = self._ensure_last_print(raw_script)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        return [self._make_response(output, channel=message.channel)]

In [13]:
class ARCSolver:

    def __init__(self, cfg, port: int = 8000):
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = ARCTemplate()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()

        self._preload_model_weights()
        self.server_process = self._start_server()

        self.client = OpenAI(
            base_url=self.base_url,
            api_key=self.api_key,
            timeout=self.cfg.session_timeout,
        )

        self._wait_for_server()
        self._initialize_kernels()

        self.notebook_start_time = time.time()
        self.problems_remaining = 50

    def _preload_model_weights(self) -> None:
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        files_to_load = []
        total_size = 0

        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)

        def _read_file(path: str) -> None:
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))

        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

    def _start_server(self) -> subprocess.Popen:
        cmd = [
            sys.executable,
            '-m',
            'vllm.entrypoints.openai.api_server',
            '--seed',
            str(self.cfg.seed),
            '--model',
            self.cfg.model_path,
            '--served-model-name',
            self.cfg.served_model_name,
            '--tensor-parallel-size',
            '1',
            '--max-num-seqs',
            str(self.cfg.batch_size),
            '--gpu-memory-utilization',
            str(self.cfg.gpu_memory_utilization),
            '--host',
            '0.0.0.0',
            '--port',
            str(self.port),
            '--dtype',
            self.cfg.dtype,
            '--kv-cache-dtype',
            self.cfg.kv_cache_dtype,
            '--max-model-len',
            str(self.cfg.context_tokens),
            '--stream-interval',
            str(self.cfg.stream_interval),
            '--async-scheduling',
            '--disable-log-stats',
            '--enable-prefix-caching',
        ]

        self.log_file = open('vllm_server.log', 'w')
        return subprocess.Popen(
            cmd,
            stdout=self.log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

    def _wait_for_server(self):
        print('Waiting for vLLM server...')
        start_time = time.time()

        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
            if return_code is not None:
                self.log_file.flush()
                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()
                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')

            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
                return
            except Exception:
                time.sleep(1)

        raise RuntimeError('Server failed to start (timeout).\n')

    def _initialize_kernels(self) -> None:
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()
        self.sandbox_pool = queue.Queue()

        def _create_sandbox():
            return ARCSandbox(timeout=self.cfg.jupyter_timeout)

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())

        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')

    def _scan_for_answer(self, text: str) -> int | None:
        boxed_matches = re.findall(r'\\boxed\s*\{\s*([0-9,]+)\s*\}', text)
        verbal_matches = re.findall(r'final\s+answer\s+is\s*([0-9,]+)', text, re.IGNORECASE)

        for matches in (boxed_matches, verbal_matches):
            if not matches:
                continue
            try:
                value = int(matches[-1].replace(',', ''))
                if 0 <= value <= 99999:
                    return value
            except ValueError:
                pass

        return None

    def _scan_verdict(self, text: str) -> str:
        match = re.search(r'VERDICT:\s*(SUPPORT|REJECT|UNSURE)', text, re.IGNORECASE)
        if not match:
            return 'UNSURE'
        return match.group(1).upper()

    def _problem_family(self, problem: str) -> str:
        text = problem.lower()
        if any(word in text for word in ['triangle', 'circle', 'angle', 'perpendicular', 'parallel', 'polygon']):
            return 'geometry'
        if any(word in text for word in ['prime', 'mod', 'divisible', 'gcd', 'lcm', 'integer', 'remainder']):
            return 'number_theory'
        if any(word in text for word in ['graph', 'color', 'permutation', 'subset', 'arrange', 'committee']):
            return 'combinatorics'
        return 'algebra'

    def _build_attempt_plan(self, problem: str) -> list[tuple[str, str, int]]:
        family = self._problem_family(problem)
        prompt_sets = {
            'geometry': [('proof', self.cfg.proof_prompt)] * 4 + [('hybrid', self.cfg.hybrid_prompt)] * 3 + [('compute', self.cfg.compute_prompt)] * 1,
            'number_theory': [('compute', self.cfg.compute_prompt)] * 4 + [('hybrid', self.cfg.hybrid_prompt)] * 2 + [('proof', self.cfg.proof_prompt)] * 2,
            'combinatorics': [('compute', self.cfg.compute_prompt)] * 3 + [('hybrid', self.cfg.hybrid_prompt)] * 3 + [('proof', self.cfg.proof_prompt)] * 2,
            'algebra': [('hybrid', self.cfg.hybrid_prompt)] * 4 + [('proof', self.cfg.proof_prompt)] * 2 + [('compute', self.cfg.compute_prompt)] * 2,
        }
        selected = prompt_sets[family][:self.cfg.attempts]
        return [(prompt, label, idx) for idx, (label, prompt) in enumerate(selected)]

    def _run_completion(
        self,
        problem: str,
        system_prompt: str,
        policy_name: str,
        attempt_index: int,
        stop_event: threading.Event,
        deadline: float,
        temperature: float | None = None,
        turn_limit: int | None = None,
    ) -> dict:
        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1,
                'Policy': policy_name,
                'Answer': None,
                'Python Calls': 0,
                'Python Errors': 0,
                'Response Length': 0,
                'Trace': '',
            }

        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        trace_parts = []
        effective_temperature = self.cfg.temperature if temperature is None else temperature
        effective_turns = self.cfg.turns if turn_limit is None else turn_limit
        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))

        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
            local_tool = ARCTool(
                local_jupyter_timeout=self.cfg.jupyter_timeout,
                tool_prompt=self.cfg.tool_prompt,
                sandbox=sandbox,
            )

            messages = self.template.apply_chat_template(system_prompt, problem, local_tool.tool_config)
            conversation = Conversation.from_messages(messages)

            for _ in range(effective_turns):
                if stop_event.is_set() or time.time() > deadline:
                    break

                prompt_ids = self.encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)
                if max_tokens < self.cfg.buffer_tokens:
                    break

                stream = self.client.completions.create(
                    model=self.cfg.served_model_name,
                    temperature=effective_temperature,
                    logprobs=self.cfg.top_logprobs,
                    max_tokens=max_tokens,
                    prompt=prompt_ids,
                    seed=attempt_seed,
                    stream=True,
                    extra_body={
                        'min_p': self.cfg.min_p,
                        'stop_token_ids': self.stop_token_ids,
                        'return_token_ids': True,
                    },
                )

                try:
                    token_buffer = []
                    text_chunks = []

                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break

                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text
                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)
                            trace_parts.append(new_text)

                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(search_text)
                            if answer is not None:
                                final_answer = answer
                                break
                finally:
                    stream.close()

                if final_answer is not None or not token_buffer:
                    break

                new_messages = self.encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]

                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    trace_parts.append('\n[final]\n' + answer_text + '\n')
                    final_answer = self._scan_for_answer(answer_text)
                    break

                if last_message.recipient == 'python':
                    python_calls += 1
                    tool_responses = local_tool.process_sync_plus(last_message)
                    response_text = tool_responses[0].content[0].text
                    trace_parts.append('\n[python]\n' + response_text + '\n')
                    if response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text:
                        python_errors += 1
                    conversation.messages.extend(tool_responses)

        except Exception:
            python_errors += 1
        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)

        return {
            'Attempt': attempt_index + 1,
            'Policy': policy_name,
            'Response Length': total_tokens,
            'Python Calls': python_calls,
            'Python Errors': python_errors,
            'Answer': final_answer,
            'Trace': ''.join(trace_parts)[-self.cfg.trace_excerpt_chars:],
        }

    def _summarize_candidates(self, detailed_results: list) -> list[dict]:
        grouped = defaultdict(list)
        for result in detailed_results:
            if result['Answer'] is not None:
                grouped[result['Answer']].append(result)

        board = []
        for answer, rows in grouped.items():
            policies = sorted({row['Policy'] for row in rows})
            python_calls = sum(row['Python Calls'] for row in rows)
            python_errors = sum(row['Python Errors'] for row in rows)
            response_tokens = sum(row['Response Length'] for row in rows)
            support_score = (
                4.0 * len(rows)
                + 2.0 * len(policies)
                + 0.10 * python_calls
                - 0.75 * python_errors
                + 0.0002 * response_tokens
            )
            best_trace = max(rows, key=lambda row: (row['Python Calls'] - row['Python Errors'], row['Response Length']))['Trace']
            board.append({
                'answer': answer,
                'votes': len(rows),
                'policy_diversity': len(policies),
                'policies': ', '.join(policies),
                'python_calls': python_calls,
                'python_errors': python_errors,
                'score': support_score,
                'trace': best_trace,
            })

        board.sort(key=lambda row: (row['votes'], row['policy_diversity'], row['score']), reverse=True)
        return board

    def _verification_prompt(self, problem: str, candidate: int) -> str:
        return (
            f'{problem}\n\n'
            f'Candidate answer to verify: {candidate}.\n'
            'Try to disprove this candidate first. If it survives, derive why it must be correct. '
            'Use Python when it can produce a contradiction, exhaustive small-case evidence, or an exact symbolic check. '
            'Do not default to agreement. End with VERDICT: SUPPORT, VERDICT: REJECT, or VERDICT: UNSURE. '
            f'If you support it, include \\boxed{{{candidate}}}. '
            f'{self.cfg.preference_prompt}'
        )

    def _verify_candidate(self, problem: str, candidate_row: dict, deadline: float) -> dict:
        supports = 0
        rejects = 0
        unsures = 0
        alternate_answers = Counter()
        traces = []

        for pass_index in range(self.cfg.verifier_passes):
            if time.time() > deadline - 5:
                break

            record = self._run_completion(
                problem=self._verification_prompt(problem, candidate_row['answer']),
                system_prompt=self.cfg.verifier_prompt,
                policy_name=f'verifier_{pass_index + 1}',
                attempt_index=100 + pass_index,
                stop_event=threading.Event(),
                deadline=deadline,
                temperature=self.cfg.verifier_temperature,
                turn_limit=self.cfg.verifier_turns,
            )

            verdict = self._scan_verdict(record['Trace'])
            if verdict == 'SUPPORT':
                supports += 1
            elif verdict == 'REJECT':
                rejects += 1
            else:
                unsures += 1

            alt = self._scan_for_answer(record['Trace'])
            if alt is not None and alt != candidate_row['answer']:
                alternate_answers[alt] += 1

            traces.append(record['Trace'])

        verifier_score = 5 * supports - 4 * rejects - unsures + candidate_row['votes'] + candidate_row['policy_diversity']
        return {
            'answer': candidate_row['answer'],
            'supports': supports,
            'rejects': rejects,
            'unsures': unsures,
            'alternate_answers': dict(alternate_answers),
            'verifier_score': verifier_score,
            'trace_excerpt': '\n\n'.join(traces)[-self.cfg.trace_excerpt_chars:],
        }

    def _judge_candidates(self, problem: str, candidates: list[dict], verification_rows: list[dict], deadline: float) -> int | None:
        if time.time() > deadline - 10:
            return None

        verification_lookup = {row['answer']: row for row in verification_rows}
        prompt_lines = [
            f'Problem: {problem}\n\n',
            'Several candidate answers survived diversified solution attempts. Choose the single most credible answer using the evidence below. '\
            'Prefer answers that have agreement across different policies and survive falsification attempts. '\
            'Reply with exactly one answer inside \\boxed{} and no extra alternatives.\n\n',
        ]

        for idx, candidate in enumerate(candidates, 1):
            verify = verification_lookup.get(candidate['answer'], {})
            prompt_lines.append(
                f'=== Candidate {idx}: {candidate["answer"]} ===\n'
                f'Votes: {candidate["votes"]}\n'
                f'Policy diversity: {candidate["policy_diversity"]} ({candidate["policies"]})\n'
                f'Python calls/errors: {candidate["python_calls"]}/{candidate["python_errors"]}\n'
                f'Verifier supports/rejects/unsures: {verify.get("supports", 0)}/{verify.get("rejects", 0)}/{verify.get("unsures", 0)}\n'
                f'Best trace excerpt:\n{candidate["trace"][:self.cfg.trace_excerpt_chars]}\n\n'
            )

        try:
            response = self.client.completions.create(
                model=self.cfg.served_model_name,
                prompt=''.join(prompt_lines),
                temperature=self.cfg.judge_temperature,
                max_tokens=self.cfg.judge_tokens,
                extra_body={'min_p': self.cfg.min_p},
            )
            chosen = self._scan_for_answer(response.choices[0].text)
            if chosen in {candidate['answer'] for candidate in candidates}:
                return chosen
        except Exception:
            pass

        return None

    def _select_answer(self, problem: str, detailed_results: list, deadline: float) -> int:
        board = self._summarize_candidates(detailed_results)
        if not board:
            print('\nFinal Answer: 0\n')
            return 0

        summary_dataframe = pd.DataFrame([
            {
                'Answer': row['answer'],
                'Votes': row['votes'],
                'Policies': row['policies'],
                'Policy Diversity': row['policy_diversity'],
                'Python Calls': row['python_calls'],
                'Python Errors': row['python_errors'],
                'Support Score': row['score'],
            }
            for row in board
        ])
        summary_dataframe = summary_dataframe.round({'Support Score': 3})
        display(summary_dataframe)

        leader = board[0]
        if len(board) == 1:
            print(f'\nFinal Answer: {leader["answer"]}\n')
            return leader['answer']

        if leader['votes'] >= self.cfg.early_stop and leader['policy_diversity'] >= 2:
            print(f'\nFinal Answer: {leader["answer"]} (early consensus across policies)\n')
            return leader['answer']

        finalists = board[:self.cfg.finalists]
        verification_rows = []
        if self.cfg.enable_verifier and time.time() <= deadline - 20:
            verification_rows = [self._verify_candidate(problem, row, deadline) for row in finalists]
            verification_df = pd.DataFrame(verification_rows)
            display(verification_df)

        verification_rows.sort(key=lambda row: row['verifier_score'], reverse=True)
        if verification_rows:
            best = verification_rows[0]
            if best['supports'] > best['rejects'] and (
                len(verification_rows) == 1 or best['verifier_score'] >= verification_rows[1]['verifier_score'] + 2
            ):
                print(f'\nFinal Answer: {best["answer"]} (verifier-selected)\n')
                return best['answer']

        judged = None
        if self.cfg.enable_judge and time.time() <= deadline - 10:
            judged = self._judge_candidates(problem, finalists, verification_rows, deadline)
        if judged is not None:
            print(f'\nFinal Answer: {judged} (judge-selected)\n')
            return judged

        fallback = board[0]['answer']
        print(f'\nFinal Answer: {fallback} (fallback support score)\n')
        return fallback

    def solve_problem(self, problem: str) -> int:
        print(f'\nProblem: {problem}\n')
        user_input = f'{problem} {self.cfg.preference_prompt}'

        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout
        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)
        deadline = time.time() + budget
        selection_reserve = min(
            self.cfg.selection_time_cap,
            max(self.cfg.selection_time_floor, budget * self.cfg.selection_time_fraction),
        )
        rollout_deadline = deadline - selection_reserve
        rollout_deadline = max(rollout_deadline, time.time() + 30)
        print(
            f'Budget: {budget:.2f} seconds | Rollout Deadline: {rollout_deadline:.2f} | Final Deadline: {deadline:.2f}\n'
        )

        tasks = self._build_attempt_plan(problem)
        detailed_results = []
        valid_answers = []
        stop_event = threading.Event()
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)

        try:
            futures = []
            for system_prompt, policy_name, attempt_index in tasks:
                future = executor.submit(
                    self._run_completion,
                    user_input,
                    system_prompt,
                    policy_name,
                    attempt_index,
                    stop_event,
                    rollout_deadline,
                )
                futures.append(future)

            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])

                    counts = Counter(valid_answers).most_common(1)
                    if (
                        len(detailed_results) >= self.cfg.min_attempts_before_stop
                        and counts
                        and counts[0][1] >= self.cfg.early_stop
                    ):
                        stop_event.set()
                        for queued in futures:
                            queued.cancel()
                        break
                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue
        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)
            self.problems_remaining = max(0, self.problems_remaining - 1)

        if detailed_results:
            results_dataframe = pd.DataFrame(detailed_results)
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')
            display(results_dataframe[['Attempt', 'Policy', 'Answer', 'Python Calls', 'Python Errors', 'Response Length']])

        if not valid_answers:
            print('\nResult: 0\n')
            return 0

        return self._select_answer(problem, detailed_results, deadline)

    def __del__(self):
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()

        if hasattr(self, 'log_file'):
            self.log_file.close()

        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
                except Exception:
                    pass

In [14]:
solver = ARCSolver(ARCConfig)

Loading model weights from /kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1 into OS Page Cache...
Processed 26 files (65.28 GB) in 85.00 seconds.

Waiting for vLLM server...
Server is ready (took 129.89 seconds).

Initializing 16 persistent Jupyter kernels...
Kernels initialized in 2.90 seconds.



In [15]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    
    id_value = id_.item(0)
    question_text = question.item(0)
    
    gc.disable()
    
    final_answer = solver.solve_problem(question_text)
    
    gc.enable()
    gc.collect()
    
    return pl.DataFrame({'id': id_value, 'answer': final_answer})

In [16]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
    
else:
    inference_server.run_local_gateway(
        ('/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    )


Problem: What is $0\times10$?

Budget: 900.00 seconds | Deadline: 1775695630.31



,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,1,119,0,0,0.655,0
1,8,123,0,0,0.670,0
2,3,134,0,0,0.747,0
3,6,139,0,0,0.801,0


,Answer,Votes,Score
0,0,4,5.605



Final Answer: 0


Problem: Solve $4+x=4$ for $x$.

Budget: 900.00 seconds | Deadline: 1775695646.69



,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,2,144,0,0,0.611,0
1,3,154,0,0,0.509,0
2,6,168,0,0,0.761,0
3,7,179,0,0,0.665,0


,Answer,Votes,Score
0,0,4,6.416



Final Answer: 0


Problem: What is $1-1$?

Budget: 900.00 seconds | Deadline: 1775695648.68



,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,7,72,0,0,0.618,0
1,1,96,0,0,0.562,0
2,4,98,0,0,0.555,0
3,8,108,0,0,0.531,0


,Answer,Votes,Score
0,0,4,7.08



Final Answer: 0

